# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Préparation obligatoire : regardez "PyTorch in 100 Seconds" pour que les sorties tensorielles ci-dessous vous semblent intuitives.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [ ]:
# Optional setup: install dependencies if they are missing in your environment.
# %pip install -q transformers torch


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "BERT understands the context of words by reading sentences in both directions."
print(sample_sentence)


In [ ]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # TODO: adjust if your sentence needs more room
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)


### Exercise 1 reflection
- [CLS] is prepended to every sequence and its final hidden state is commonly used as an aggregate representation of the whole sentence (e.g., for classification). [SEP] marks the end of a sentence (or separates two sentences in pair tasks like question answering), telling the encoder where one segment stops.
- The attention mask is a binary vector (1 for real tokens, 0 for [PAD] tokens). It is added (as a large negative bias) to the self-attention scores before the softmax, so padded positions get an attention weight of essentially zero and cannot influence the representations of real tokens.


## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [ ]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "I absolutely loved this course, it explained everything so clearly!"
prediction = sentiment_pipeline(sentence)
prediction


### Exercise 2 reflection
- Yes, the predicted label is POSITIVE, which matches expectations since the sentence contains clearly positive words ("loved", "clearly").
- The confidence score is typically above 0.99 for such an unambiguous sentence, meaning the model is highly certain about its prediction. A score close to 1.0 indicates low ambiguity in the input text.


## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.max_length = max_length
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        text = text.strip()
        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {k: v.to(self.device) for k, v in encoding.items()}

    def predict(self, text: str) -> Dict[str, float]:
        inputs = self.preprocess(text)
        with torch.no_grad():
            outputs = self.model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]
        pred_id = int(torch.argmax(probs))
        label = self.model.config.id2label[pred_id]
        return {"label": label, "probability": float(probs[pred_id])}


In [ ]:
analyzer = BERTSentimentAnalyzer()
samples = [
    "This is the best product I have ever bought!",
    "I am really disappointed with the service."
]
for text in samples:
    print(text)
    print(analyzer.predict(text))


## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()
        self.id2label = self.model.config.id2label

    def recognize(self, text: str):
        encoding = self.tokenizer(text, return_tensors="pt", return_offsets_mapping=True, truncation=True)
        offsets = encoding.pop("offset_mapping")[0].tolist()
        inputs = {k: v.to(self.device) for k, v in encoding.items()}

        with torch.no_grad():
            outputs = self.model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)[0].tolist()

        entities = []
        current = None
        for pred_id, (start, end) in zip(predictions, offsets):
            if start == end:
                continue
            label = self.id2label[pred_id]
            if label == "O":
                if current:
                    entities.append(current)
                    current = None
                continue
            tag_type = label.split("-")[-1]
            is_begin = label.startswith("B-")
            if current and not is_begin and current["entity"] == tag_type and current["end"] == start:
                current["end"] = end
                current["text"] = text[current["start"]:current["end"]]
            else:
                if current:
                    entities.append(current)
                current = {"text": text[start:end], "entity": tag_type, "start": start, "end": end}
        if current:
            entities.append(current)
        return entities


In [ ]:
ner = BERTNamedEntityRecognizer()
sample_text = "Marie Curie worked at the University of Paris and won the Nobel Prize."
ner.recognize(sample_text)


## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

Fill the table with concise statements (one line each).

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | Transformer encoder only, bidirectional self-attention | Transformer decoder only, causal (unidirectional) self-attention |
| Primary purpose | Language understanding (build rich contextual representations) | Language generation (predict the next token) |
| Typical use cases | Classification, NER, question answering, sentence embeddings | Text generation, chatbots, summarization, code generation |
| Strengths | Strong contextual understanding from both directions, great for fine-tuning on understanding tasks | Excellent at fluent, coherent free-form text generation; scales well with size |
| Weaknesses | Cannot natively generate text token-by-token; needs task-specific heads | Only sees left context, so it can be weaker at deep bidirectional understanding tasks |


## Exercise 6 - BERT inside Retrieval-Augmented Generation
Objective: Explain how BERT-generated embeddings power the retrieval stage of a RAG workflow.

Address each bullet with a short paragraph:

1. **Encoding queries and documents:** BERT (or a BERT-derived sentence encoder such as Sentence-BERT) passes a query or a document chunk through its transformer layers and pools the resulting token representations (often the [CLS] vector or a mean-pooled vector) into a single dense embedding that captures the semantic meaning of the text.

2. **Storing and searching embeddings:** These dense vectors are stored in a vector database (e.g., FAISS, Pinecone, Weaviate, Chroma). At query time, the query is encoded with the same model, and an approximate nearest-neighbor search (typically cosine similarity or dot product) retrieves the document embeddings that are closest to the query embedding, i.e., the most semantically relevant passages.

3. **Handing results to the generator:** The top-k retrieved passages are concatenated with the original user query into a prompt, which is then fed to a generative model like GPT. The generator uses this retrieved context as grounding evidence to produce a more accurate, up-to-date, and factually supported answer instead of relying purely on its parametric memory.

4. **Concrete application example:** A customer support chatbot for a software company can use BERT embeddings to retrieve the most relevant pages from internal documentation or past support tickets, then have GPT generate a natural-language answer grounded in that retrieved content, reducing hallucinations and keeping answers up to date without retraining the generative model.
